# Qwen2.5-7B-Instruct: serving BF16 vs FP8 with vLLM on an L4

**Session 2 of 2 — vLLM 0.7.3 only** (Python 3.12 runtime; vLLM 0.7.3 does not install on 3.13).

1. Sanity-check the FP8 checkpoint in vLLM
2. Concurrent load test against the OpenAI-compatible server: TTFT, inter-token latency (ITL),
   p50/p95/p99, throughput at 1 / 10 / 50 users, on short and long prompts
3. Single-request prompt-length check
4. Offline batch throughput (engine ceiling, no server)
5. Cost per million output tokens

> Outputs are from the recorded run; install logs and engine logs were stripped. The load-test cells
> were re-run several times while switching model and workload, so their final results are collected
> in the table at the end rather than as cell outputs.

## 0. Install, restart, verify

In [ ]:
!pip install vllm==0.7.3 -q
!pip install "transformers>=4.48.2,<5.0" -q
!pip uninstall torchvision -y
# import os; os.kill(os.getpid(), 9)

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'torchvision', '-y'], capture_output=True)

from vllm import LLM, SamplingParams   # first CUDA import
import vllm, torch, time, gc, transformers
print(f"vLLM {vllm.__version__} | torch {torch.__version__} | transformers {transformers.__version__}")

vLLM 0.7.3 | torch 2.5.1+cu124 | transformers 4.57.6


In [ ]:
from google.colab import drive; drive.mount('/content/drive')

MODEL_ID     = "Qwen/Qwen2.5-7B-Instruct"
MODEL_FP8    = "/content/drive/MyDrive/qwen2.5-7b-fp8"

## 1. Sanity check: does the FP8 checkpoint generate coherent text?

In [2]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

import gc, torch; gc.collect(); torch.cuda.empty_cache()
tokenizer = AutoTokenizer.from_pretrained(MODEL_FP8)
llm = LLM(model=MODEL_FP8, gpu_memory_utilization=0.85, max_model_len=4096)

messages = [{"role": "user", "content": "Explain what FP8 quantization does in one sentence."}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

sampling_params = SamplingParams(max_tokens=100, temperature=0.7)
outputs = llm.generate([text], sampling_params)
print(outputs[0].outputs[0].text)

# What to look for:
#  - Coherent, on-topic English sentence(s)               -> good, checkpoint is healthy
#  - Empty output, repeated garbage tokens, or gibberish   -> something broke in quantization,
#    stop here and debug before running the full CELL E perplexity eval or any load testing

del llm
import gc, torch; gc.collect(); torch.cuda.empty_cache()

FP8 quantization reduces the precision of floating-point numbers to a smaller format, specifically 8 bits, while trying to preserve the dynamic range and precision of the original data.


## 2. Start the OpenAI-compatible server (FP8; swap in `MODEL_ID` for the BF16 run)

In [3]:
import subprocess, time, requests

def start_server(model_path, port=8000, timeout_s=600):
    proc = subprocess.Popen([
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", model_path,
        "--port", str(port),
        "--gpu-memory-utilization", "0.85",
        "--max-model-len", "4096",
    ], start_new_session=True)   # lets Cell H stop the server AND its engine process

    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if proc.poll() is not None:
            raise RuntimeError(
                f"Server process exited early with code {proc.returncode}. "
                "Scroll up for the actual vLLM error -- it prints directly."
            )
        try:
            r = requests.get(f"http://localhost:{port}/health")
            if r.status_code == 200:
                print("Server ready.")
                return proc
        except Exception:
            pass
        time.sleep(2)
    raise RuntimeError("Server did not start in time.")

# server_proc = start_server(MODEL_ID, port=8000)  # baseline instead
server_proc = start_server(MODEL_FP8, port=8000)

Server ready.


In [ ]:
!pip install nest_asyncio -q
import nest_asyncio
nest_asyncio.apply()

## 3. Concurrent load test
A streaming async client timestamps the first token separately, so TTFT and ITL are measured per request.

In [ ]:
import asyncio, aiohttp, time, statistics, requests, random

# Two traffic shapes. "short": chat-style prompts of ~10-15 tokens.
# "long": a shared technical passage repeated to ~250 / 1,000 / 2,000 tokens plus a question
# (prefill-heavy, closer to production traffic with system prompts / RAG context).
SHORT_PROMPTS = [
    "Explain the difference between INT8 and FP8 quantization.",
    "Write a short summary of how PagedAttention works in vLLM.",
    "What are the tradeoffs of mixed-precision inference?",
    "Describe continuous batching in LLM serving.",
] * 15

random.seed(0)
_ctx = ("FP8 quantization stores weights in 8-bit floating point with per-channel scales, "
        "reducing memory traffic during decode while preserving dynamic range. ")
LONG_PROMPTS = [
    _ctx * random.choice([10, 40, 80]) + "\n\nQuestion: " + q
    for q in ["Summarize the key idea.", "What are the trade-offs?",
              "How does this affect serving cost?", "Explain it to a beginner."]
    for _ in range(65)
]

WORKLOAD = "short"   # "short" or "long"
PROMPTS = SHORT_PROMPTS if WORKLOAD == "short" else LONG_PROMPTS

In [ ]:
# Ask the server what it's actually serving -- vLLM rejects requests where
# "model" doesn't exactly match, and this stays correct whether the server
# is currently running MODEL_FP8 or MODEL_ID (baseline), no manual editing.
SERVED_MODEL = requests.get("http://localhost:8000/v1/models").json()["data"][0]["id"]
print(f"Serving: {SERVED_MODEL}")

async def send_request_streaming(session, prompt, url):
    """ Streams the response so we can timestamp the first token separately
    from total completion time -> gives TTFT, not just end-to-end latency. """
    payload = {
        "model": SERVED_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 128,
        "stream": True,
    }
    start = time.perf_counter()
    ttft = None
    token_count = 0
    error = None
    async with session.post(url, json=payload) as resp:
        if resp.status != 200:
            error = f"HTTP {resp.status}: {(await resp.text())[:300]}"
        else:
            async for line in resp.content:
                decoded = line.decode("utf-8").strip()
                if not decoded or not decoded.startswith("data:"):
                    continue
                data = decoded[len("data:"):].strip()
                if data == "[DONE]":
                    break
                if ttft is None:
                    ttft = time.perf_counter() - start  # timestamp of first content chunk
                token_count += 1
    total_time = time.perf_counter() - start
    itl = (total_time - ttft) / max(token_count - 1, 1) if ttft else None
    return {"ttft": ttft, "total_time": total_time, "itl": itl,
            "token_count": token_count, "error": error}

async def run_load_test(concurrency, url="http://localhost:8000/v1/chat/completions"):
    async with aiohttp.ClientSession() as session:
        sem = asyncio.Semaphore(concurrency)
        async def bound_request(p):
            async with sem:
                return await send_request_streaming(session, p, url)
        start = time.perf_counter()
        results = await asyncio.gather(*[bound_request(p) for p in PROMPTS[:concurrency * 5]])
        wall_time = time.perf_counter() - start
        return results, wall_time

def percentile(values, pct):
    if not values:
        return None
    values = sorted(values)
    return values[int(len(values) * pct)]

def fmt(v, suffix="s", decimals=3):
    return f"{v:.{decimals}f}{suffix}" if v is not None else "N/A"

In [ ]:
for c in [1, 10, 50]:
    results, wall_time = asyncio.run(run_load_test(c))

    errors = [r["error"] for r in results if r["error"]]
    if errors:
        print(f"Concurrency={c:3d} -- {len(errors)}/{len(results)} requests errored. First error: {errors[0]}")
        continue

    ttfts = [r["ttft"] for r in results if r["ttft"] is not None]
    totals = [r["total_time"] for r in results]
    itls = [r["itl"] for r in results if r["itl"] is not None]
    req_throughput = len(results) / wall_time
    total_tokens = sum(r["token_count"] for r in results)
    system_tps = total_tokens / wall_time
    per_user_tps = statistics.mean(1 / i for i in itls) if itls else None

    print(f"Concurrency={c:3d}")
    print(f"  TTFT   p50={fmt(percentile(ttfts,0.50))} p95={fmt(percentile(ttfts,0.95))} p99={fmt(percentile(ttfts,0.99))}")
    print(f"  Total  p50={fmt(percentile(totals,0.50),decimals=2)} p95={fmt(percentile(totals,0.95),decimals=2)} p99={fmt(percentile(totals,0.99),decimals=2)}")
    print(f"  ITL(avg per-token decode) ~{fmt(statistics.mean(itls)*1000 if itls else None, 'ms', 1)}")
    print(f"  Request throughput: {req_throughput:.2f} req/s")
    print(f"  System TPS (aggregate output tokens/sec): {system_tps:.1f}")
    print(f"  Per-user TPS (tokens/sec, one stream): {fmt(per_user_tps, '', 1)}")

## 4. Prompt length vs latency (single requests)
Recorded against the BF16 server: TTFT grows with prompt length, ITL stays flat (~58 ms).

In [ ]:
import asyncio, aiohttp

async def send_single(prompt, url="http://localhost:8000/v1/chat/completions"):
    async with aiohttp.ClientSession() as session:
        return await send_request_streaming(session, prompt, url)  # reuses CELL G's function

_base = ("Explain FP8 quantization in detail, covering how scaling factors are computed, "
         "how it differs from INT8, what hardware supports it, and why it matters for LLM "
         "inference throughput and memory usage. ")

PROMPT_VARIANTS = {
    "short (~20 tok)":   "Explain FP8 quantization briefly.",
    "medium (~200 tok)": _base * 3,
    "long (~500 tok)":   _base * 8,
}

for label, prompt in PROMPT_VARIANTS.items():
    result = asyncio.run(send_single(prompt))
    print(f"{label}: TTFT={result['ttft']:.3f}s | Total={result['total_time']:.2f}s | "
          f"ITL={result['itl']*1000:.1f}ms")

## 5. Stop the server and free the GPU

In [ ]:
# terminate() only stops the API server parent; the engine process keeps the GPU memory.
# The server was started with start_new_session=True, so stop the whole process group.
import os, signal
os.killpg(os.getpgid(server_proc.pid), signal.SIGTERM)
server_proc.wait()
!nvidia-smi --query-gpu=memory.used --format=csv

## 6. Offline batch throughput (no server)
240 short prompts submitted at once. The recorded output below is the **BF16** run (`MODEL_ID`); FP8 reached 2,298 tok/s.

In [ ]:
PROMPTS = SHORT_PROMPTS  # 60 short prompts; x4 below = 240

In [4]:
from vllm import LLM, SamplingParams
import time

llm = LLM(model=MODEL_ID, gpu_memory_utilization=0.85, max_model_len=4096)  # MODEL_ID = BF16 baseline; MODEL_FP8 for the FP8 run
sampling_params = SamplingParams(max_tokens=128)

batch_prompts = PROMPTS * 4  # large batch to saturate the GPU

start = time.perf_counter()
outputs = llm.generate(batch_prompts, sampling_params)
elapsed = time.perf_counter() - start

total_output_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
print(f"Offline batch: {len(batch_prompts)} prompts, {total_output_tokens} output tokens, "
      f"{elapsed:.2f}s -> {total_output_tokens/elapsed:.2f} tokens/sec (no server overhead)")

Offline batch: 240 prompts, 30691 output tokens, 15.20s -> 2019.26 tokens/sec (no server overhead)


## 7. Cost per million output tokens
L4 on-demand at an assumed $0.70/hr; throughput from the 50-user short-prompt load test.

In [5]:
# L4 on-demand pricing ~$0.70/hr (GCP on-demand rate) -- verify current rate
# if this matters for your final writeup, cloud pricing changes.
GPU_HOURLY_COST = 0.70

def cost_per_million_tokens(tokens_per_sec, gpu_hourly_cost=GPU_HOURLY_COST):
    tokens_per_hour = tokens_per_sec * 3600
    cost_per_token = gpu_hourly_cost / tokens_per_hour
    return cost_per_token * 1_000_000

# Measured system TPS at concurrency=50 (sustained load, most representative
# of real production traffic vs the concurrency=1 idle-GPU case):
results = {
    "baseline_bf16": {"tokens_per_sec": 434.9},
    "fp8":           {"tokens_per_sec": 605.1},
}

for name, r in results.items():
    cost = cost_per_million_tokens(r["tokens_per_sec"])
    print(f"{name}: ${cost:.3f} per 1M tokens")

baseline_cost = cost_per_million_tokens(results["baseline_bf16"]["tokens_per_sec"])
fp8_cost = cost_per_million_tokens(results["fp8"]["tokens_per_sec"])
savings_pct = (1 - fp8_cost / baseline_cost) * 100
print(f"\nFP8 reduces cost per token by {savings_pct:.1f}% at sustained (concurrency=50) load.")

baseline_bf16: $0.447 per 1M tokens
fp8: $0.321 per 1M tokens
FP8 reduces cost per token by 28.1% at sustained (concurrency=50) load.


## Recorded results
The load tests were run once per model (FP8 checkpoint, then `Qwen/Qwen2.5-7B-Instruct` in BF16) and
per workload, by switching `start_server(...)` and `WORKLOAD` above; per-run logs are summarized in
[`docs/report.md`](../docs/report.md).

**Short prompts** (60 prompts, `max_tokens=128`)

| Users | Model | TTFT p50 | ITL | System throughput |
|---:|---|---:|---:|---:|
| 1 | BF16 | 0.125 s | 57.9 ms | 17.1 tok/s |
| 1 | FP8 | 0.081 s | 37.0 ms | 26.8 tok/s |
| 10 | BF16 | 0.206 s | 61.0 ms | 161.0 tok/s |
| 10 | FP8 | 0.143 s | 40.3 ms | 235.4 tok/s |
| 50 | BF16 | 0.660 s | 70.2 ms | 434.9 tok/s |
| 50 | FP8 | 0.690 s | 49.5 ms | 605.1 tok/s |

**Long prompts** (~250–2,000 tokens, `max_tokens=128`)

| Users | Model | TTFT p50 | ITL | System throughput |
|---:|---|---:|---:|---:|
| 1 | BF16 | 0.351 s | 57.4 ms | 15.9 tok/s |
| 1 | FP8 | 0.371 s | 36.9 ms | 22.9 tok/s |
| 10 | BF16 | 0.644 s | 111.1 ms | 77.0 tok/s |
| 10 | FP8 | 0.716 s | 95.5 ms | 86.9 tok/s |
| 50 | BF16 | 4.204 s | 218.5 ms | 193.7 tok/s |
| 50 | FP8 | 1.647 s | 231.8 ms | 213.0 tok/s |